In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union
import glob
import os

In [2]:
pd.set_option('display.max_rows', 500)

In [3]:
from vista_criticality_utils import *

In [ ]:
# Reload modules
import importlib
import vista_criticality_utils
importlib.reload(vista_criticality_utils)
from vista_criticality_utils import *

# 1. Load assets to a new graph

In [5]:
asset_file = '/home/vagrant/Documents/vista/data-science/data/02-fluvial-flooding.csv'

In [6]:
incident_data = pd.read_csv(asset_file)

In [7]:
incident_data.head(20)

,Asset_ID,Asset_Type,Asset_Name,lat,long,criticality_score,backup_asset_id
0,BSC01,Business/Commercial,IndustrialSpace,50.701337,-1.298019,1,BSC02
1,BSC02,Business/Commercial,IndustrialSpace,50.662767,-1.150154,1,BSC03
2,BSC03,Business/Commercial,IndustrialSpace,50.757944,-1.290132,1,NaN
3,BSC04,Business/Commercial,IndustrialSpace,50.637089,-1.179144,1,NaN
4,BSC05,Business/Commercial,IndustrialSpace,50.591618,-1.255649,1,NaN
5,BSC06,Business/Commercial,RetailKiosk,50.718441,-1.158219,1,NaN
6,BSC07,Business/Commercial,RetailKiosk,50.689932,-1.088592,1,NaN
7,BSC08,Business/Commercial,RetailKiosk,50.699454,-1.295762,1,NaN
8,BSC09,Business/Commercial,RetailKiosk,50.758017,-1.287798,1,NaN
9,BSC10,Business/Commercial,RetailKiosk,50.726535,-1.132907,1,NaN


In [8]:
incident_data.shape

(133, 7)

In [9]:
incident_data.Asset_ID.nunique()

133

In [10]:
#incident_data.Asset_ID.duplicated()

In [11]:
G = load_assets_to_graph(assets_filepath=asset_file)

Successfully loaded /home/vagrant/Documents/vista/data-science/data/02-fluvial-flooding.csv
Added 133 nodes to the graph.


In [12]:
dependency_file = '/home/vagrant/Documents/vista/data-science/data/02-dependency-mapping.csv'
asset_dependencies = pd.read_csv(dependency_file)

In [13]:
asset_dependencies.head(10)

,from_asset,to_asset,connection,dependency_score
0,PGS07,BSC04,PGS07-->BSC04,2
1,PGS07,TRR03,PGS07-->TRR03,2
2,PGS07,NH02,PGS07-->NH02,2
3,PGS07,TEL03,PGS07-->TEL03,2
4,PGS07,VH05,PGS07-->VH05,2
5,PGS07,NH04,PGS07-->NH04,2
6,PGS07,FS07,PGS07-->FS07,2
7,PGS04,FS03,PGS04-->FS03,2
8,PGS04,BSC02,PGS04-->BSC02,2
9,PGS04,SCH02,PGS04-->SCH02,2


In [15]:
G.nodes['TRF02']

{'Asset_ID': 'TRF02',
 'Asset_Type': 'Transport',
 'Asset_Name': 'FerryPort',
 'lat': 50.763106,
 'long': -1.2960338,
 'criticality_score': 2,
 'backup_asset_id': 'TRA01',
 'status': 'working'}

In [16]:
G = add_dependencies_to_graph(graph=G, dependencies_filepath=dependency_file)


Successfully loaded /home/vagrant/Documents/vista/data-science/data/02-dependency-mapping.csv
Added edges. The graph now has 124 edges.


In [17]:
G.number_of_edges()

124

In [18]:
## Add dependency score aS node attributes
G = compute_dependency_score(G=G)

In [19]:
G.nodes['TRF02']

{'Asset_ID': 'TRF02',
 'Asset_Type': 'Transport',
 'Asset_Name': 'FerryPort',
 'lat': 50.763106,
 'long': -1.2960338,
 'criticality_score': 2,
 'backup_asset_id': 'TRA01',
 'status': 'working',
 'dependency_score': 0}

In [20]:
incident_data = graph_to_dataframe(G)

In [21]:
incident_data.head(20)

,Asset_ID,Asset_Type,Asset_Name,lat,long,criticality_score,backup_asset_id,status,dependency_score
0,BSC01,Business/Commercial,IndustrialSpace,50.701337,-1.298019,1,BSC02,working,0
1,BSC02,Business/Commercial,IndustrialSpace,50.662767,-1.150154,1,BSC03,working,0
2,BSC03,Business/Commercial,IndustrialSpace,50.757944,-1.290132,1,NaN,working,0
3,BSC04,Business/Commercial,IndustrialSpace,50.637089,-1.179144,1,NaN,working,0
4,BSC05,Business/Commercial,IndustrialSpace,50.591618,-1.255649,1,NaN,working,0
5,BSC06,Business/Commercial,RetailKiosk,50.718441,-1.158219,1,NaN,working,0
6,BSC07,Business/Commercial,RetailKiosk,50.689932,-1.088592,1,NaN,working,0
7,BSC08,Business/Commercial,RetailKiosk,50.699454,-1.295762,1,NaN,working,0
8,BSC09,Business/Commercial,RetailKiosk,50.758017,-1.287798,1,NaN,working,0
9,BSC10,Business/Commercial,RetailKiosk,50.726535,-1.132907,1,NaN,working,0


## New Exposure Score

In [22]:
flood_layers = '/home/vagrant/Documents/vista/data-science/data/New_exposure_score_data/flood_files'
heat_stress_layers = '/home/vagrant/Documents/vista/data-science/data/New_exposure_score_data/heat_stress_files'
landslide_layers = '/home/vagrant/Documents/vista/data-science/data/New_exposure_score_data/landslide_files'

In [23]:
# Construct the search pattern to find all .geojson files in the specified folder
flood_geojson_search_pattern = os.path.join(flood_layers, '*.geojson')
flood_geojson_file_paths = glob.glob(flood_geojson_search_pattern)

heat_geojson_search_pattern = os.path.join(heat_stress_layers, '*.geojson')
heat_geojson_file_paths = glob.glob(heat_geojson_search_pattern)

landslide_geojson_search_pattern = os.path.join(landslide_layers, '*.geojson')
landslide_geojson_file_paths = glob.glob(landslide_geojson_search_pattern)

In [24]:
incident_data = compute_exposure_score_v2(assets_df=incident_data, flood_files=flood_geojson_file_paths)

In [28]:
incident_data.head(30)

,Asset_ID,Asset_Type,Asset_Name,lat,long,criticality_score,backup_asset_id,status,dependency_score,exposure_score
0,BSC01,Business/Commercial,IndustrialSpace,50.701337,-1.298019,1,BSC02,working,0,2
1,BSC02,Business/Commercial,IndustrialSpace,50.662767,-1.150154,1,BSC03,working,0,2
2,BSC03,Business/Commercial,IndustrialSpace,50.757944,-1.290132,1,NaN,working,0,0
3,BSC04,Business/Commercial,IndustrialSpace,50.637089,-1.179144,1,NaN,working,0,2
4,BSC05,Business/Commercial,IndustrialSpace,50.591618,-1.255649,1,NaN,working,0,2
5,BSC06,Business/Commercial,RetailKiosk,50.718441,-1.158219,1,NaN,working,0,0
6,BSC07,Business/Commercial,RetailKiosk,50.689932,-1.088592,1,NaN,working,0,1
7,BSC08,Business/Commercial,RetailKiosk,50.699454,-1.295762,1,NaN,working,0,2
8,BSC09,Business/Commercial,RetailKiosk,50.758017,-1.287798,1,NaN,working,0,0
9,BSC10,Business/Commercial,RetailKiosk,50.726535,-1.132907,1,NaN,working,0,0


In [29]:
incident_data.to_csv('exposure_score_with_flood_layer.csv', index=False)

In [30]:
incident_data_with_heat_stress_layer = compute_exposure_score_v2(assets_df=incident_data, flood_files=flood_geojson_file_paths, heat_files=heat_geojson_file_paths)

In [31]:
incident_data_with_heat_stress_layer.head(20)

,Asset_ID,Asset_Type,Asset_Name,lat,long,criticality_score,backup_asset_id,status,dependency_score,exposure_score
0,BSC01,Business/Commercial,IndustrialSpace,50.701337,-1.298019,1,BSC02,working,0,2
1,BSC02,Business/Commercial,IndustrialSpace,50.662767,-1.150154,1,BSC03,working,0,3
2,BSC03,Business/Commercial,IndustrialSpace,50.757944,-1.290132,1,NaN,working,0,0
3,BSC04,Business/Commercial,IndustrialSpace,50.637089,-1.179144,1,NaN,working,0,2
4,BSC05,Business/Commercial,IndustrialSpace,50.591618,-1.255649,1,NaN,working,0,2
5,BSC06,Business/Commercial,RetailKiosk,50.718441,-1.158219,1,NaN,working,0,0
6,BSC07,Business/Commercial,RetailKiosk,50.689932,-1.088592,1,NaN,working,0,1
7,BSC08,Business/Commercial,RetailKiosk,50.699454,-1.295762,1,NaN,working,0,2
8,BSC09,Business/Commercial,RetailKiosk,50.758017,-1.287798,1,NaN,working,0,0
9,BSC10,Business/Commercial,RetailKiosk,50.726535,-1.132907,1,NaN,working,0,0


In [89]:
incident_data_with_heat_stress_layer.to_csv('exposure_score_with_flood_heat_layer.csv', index=False)

In [91]:
incident_data_with_heat_stress_landslide_layer= compute_exposure_score_v2(assets_df=incident_data, flood_files=flood_geojson_file_paths, heat_files=heat_geojson_file_paths, landslide_files=landslide_geojson_file_paths)

In [92]:
incident_data_with_heat_stress_landslide_layer.head(20)

,Asset_ID,Asset_Type,Asset_Name,lat,long,criticality_score,backup_asset_id,status,dependency_score,exposure_score
0,WS01,WaterSupply,WaterReservoir,50.678504,-1.152067,3,WS02,working,0,3
1,SW01,Sewage,WastewaterCollectionNetwork,50.739617,-1.328310,3,SW02,working,0,0
2,SW02,Sewage,WastewaterCollectionNetwork,50.685999,-1.543561,3,SW03,working,0,0
3,SW03,Sewage,WastewaterCollectionNetwork,50.667008,-1.184479,3,NaN,working,0,2
4,SW04,Sewage,WastewaterCollectionNetwork,50.761405,-1.288696,3,NaN,working,0,0
5,PGS01,PowerGen/Substation,LowVoltageElectricitySubstationComplex,50.719325,-1.202044,3,PGS02,working,3,1
6,PGS02,PowerGen/Substation,HighVoltageElectricitySubstationComplex,50.710424,-1.253947,3,PGS03,working,0,0
7,PGS03,PowerGen/Substation,LowVoltageElectricitySubstationComplex,50.745720,-1.285433,3,NaN,working,3,0
8,PGS04,PowerGen/Substation,LowVoltageElectricitySubstationComplex,50.662086,-1.153749,3,NaN,working,0,3
9,WS02,WaterSupply,WaterDistributionComplex,50.618912,-1.186271,3,WS03,working,0,1


In [93]:
incident_data_with_heat_stress_landslide_layer.to_csv('exposure_score_with_flood_heat_landslide_layer.csv',index=False)

## Redundancy Score

In [94]:
incident_data = compute_redundancy_score(df=incident_data)

In [95]:
incident_data.head(20)

,Asset_ID,Asset_Type,Asset_Name,lat,long,criticality_score,backup_asset_id,status,dependency_score,exposure_score,redundancy_score
0,WS01,WaterSupply,WaterReservoir,50.678504,-1.152067,3,WS02,working,0,3,2
1,SW01,Sewage,WastewaterCollectionNetwork,50.739617,-1.328310,3,SW02,working,0,0,2
2,SW02,Sewage,WastewaterCollectionNetwork,50.685999,-1.543561,3,SW03,working,0,0,2
3,SW03,Sewage,WastewaterCollectionNetwork,50.667008,-1.184479,3,NaN,working,0,2,3
4,SW04,Sewage,WastewaterCollectionNetwork,50.761405,-1.288696,3,NaN,working,0,0,3
5,PGS01,PowerGen/Substation,LowVoltageElectricitySubstationComplex,50.719325,-1.202044,3,PGS02,working,3,1,1
6,PGS02,PowerGen/Substation,HighVoltageElectricitySubstationComplex,50.710424,-1.253947,3,PGS03,working,0,0,1
7,PGS03,PowerGen/Substation,LowVoltageElectricitySubstationComplex,50.745720,-1.285433,3,NaN,working,3,0,3
8,PGS04,PowerGen/Substation,LowVoltageElectricitySubstationComplex,50.662086,-1.153749,3,NaN,working,0,3,3
9,WS02,WaterSupply,WaterDistributionComplex,50.618912,-1.186271,3,WS03,working,0,1,0


In [101]:
incident_data.to_csv('incident_data_with_all_scores.csv', index=False)

### Add exposure and redundancy scores as graph node addtributes

In [96]:
G = add_attribute_from_df(G=G, df=incident_data, column_name='exposure_score')

Added exposure_score to the graph.


In [97]:
G.nodes['TRF02']

{'Asset_ID': 'TRF02',
 'Asset_Type': 'Transport-Ferry',
 'Asset_Name': 'FerryPort',
 'lat': 50.763106,
 'long': -1.2960338,
 'criticality_score': 3,
 'backup_asset_id': 'TRF01',
 'status': 'working',
 'dependency_score': 3,
 'exposure_score': 0}

In [98]:
G = add_attribute_from_df(G=G, df=incident_data, column_name='redundancy_score')

Added redundancy_score to the graph.


In [99]:
G.nodes['TRF02']

{'Asset_ID': 'TRF02',
 'Asset_Type': 'Transport-Ferry',
 'Asset_Name': 'FerryPort',
 'lat': 50.763106,
 'long': -1.2960338,
 'criticality_score': 3,
 'backup_asset_id': 'TRF01',
 'status': 'working',
 'dependency_score': 3,
 'exposure_score': 0,
 'redundancy_score': 0}

In [111]:
G.nodes['PGS01']

{'Asset_ID': 'PGS01',
 'Asset_Type': 'PowerGen/Substation',
 'Asset_Name': 'LowVoltageElectricitySubstationComplex',
 'lat': 50.719325,
 'long': -1.2020439,
 'criticality_score': 3,
 'backup_asset_id': 'PGS02',
 'status': 'working',
 'dependency_score': 3,
 'exposure_score': 1,
 'redundancy_score': 1}

## Compute Asset Score

In [112]:
G = compute_asset_score(G=G)

Added asset_score to assets.


In [113]:
G.nodes['TRF02']

{'Asset_ID': 'TRF02',
 'Asset_Type': 'Transport-Ferry',
 'Asset_Name': 'FerryPort',
 'lat': 50.763106,
 'long': -1.2960338,
 'criticality_score': 3,
 'backup_asset_id': 'TRF01',
 'status': 'working',
 'dependency_score': 3,
 'exposure_score': 0,
 'redundancy_score': 0,
 'asset_score': 6}

## Simulate failure

In [115]:
failed_nodes = simulate_failure(G=G, failed_node='TRF01')

Failing: TRF01
Propagated failure to: JN18 (via dependency 2.0)
Propagated failure to: JN12 (via dependency 2.0)
Propagated failure to: BSC09 (via dependency 2.0)
Propagated failure to: HS01 (via dependency 2.0)
Propagated failure to: JN15 (via dependency 2.0)
Propagated failure to: PH06 (via dependency 2.0)
Propagated failure to: JN13 (via dependency 2.0)
Propagated failure to: JN24 (via dependency 2.0)
Propagated failure to: JN05 (via dependency 2.0)
Propagated failure to: JN11 (via dependency 2.0)


In [116]:
failed_nodes

['JN18',
 'JN12',
 'BSC09',
 'HS01',
 'JN15',
 'PH06',
 'JN13',
 'JN24',
 'JN05',
 'JN11']

In [123]:
check_node_and_edges(G, 'TRF01')


--- Analysis for Node: TRF01 ---

[Asset Attributes]
  - Asset_ID: TRF01
  - Asset_Type: Transport-Ferry
  - Asset_Name: FerryPort
  - lat: 50.759141
  - long: -1.2905877
  - criticality_score: 3
  - backup_asset_id: TRF02
  - status: failed
  - dependency_score: 2
  - exposure_score: 0
  - redundancy_score: 0
  - asset_score: 5

[Outgoing Dependencies (This asset depends on...)]
  -> TRF01 impacts JN05 with dependency_score (weight): 2.0
  -> TRF01 impacts HS01 with dependency_score (weight): 2.0
  -> TRF01 impacts PH06 with dependency_score (weight): 2.0
  -> TRF01 impacts BSC09 with dependency_score (weight): 2.0

[Incoming Dependencies (...depend on this asset)]
  <- PGS03 is parent to TRF01 with dependency_score (weight): 3.0
------------------------------


##### Finished

In [109]:
attributes_to_remove = ['asset_resilience_score']
G = remove_node_attributes(G, attributes_to_remove)

Attempted to remove the following attributes from all nodes: asset_resilience_score


In [110]:
G.nodes['TRF02']

{'Asset_ID': 'TRF02',
 'Asset_Type': 'Transport-Ferry',
 'Asset_Name': 'FerryPort',
 'lat': 50.763106,
 'long': -1.2960338,
 'criticality_score': 3,
 'backup_asset_id': 'TRF01',
 'status': 'working',
 'dependency_score': 3,
 'exposure_score': 0,
 'redundancy_score': 0}

In [18]:
new_asset_file = '/home/akutekwea/VISTA-Development/scenario_data/01-additional-new-assets.csv'
new_asset_mapping_file = '/home/akutekwea/VISTA-Development/scenario_data/01-additional-new-assets-mapping.csv'

In [6]:
new_assets = pd.read_csv(new_asset_file)
new_dependencies = pd.read_csv(new_asset_mapping_file)

In [11]:
new_assets.head()

,Asset_ID,Asset_Type,Asset_Name,lat,long,criticality_score,backup_asset_id
0,PGS09,PowerGen/Substation,LowVoltageElectricitySubstationComplex,50.703494,-1.292065,3,NaN
1,LFS01,LiquidFuelStorage,LiquidFuelStorageComplex,50.748784,-1.288094,2,LFS02
2,LFS02,LiquidFuelStorage,LiquidFuelStorageComplex,50.745210,-1.287311,2,LFS03
3,LFS03,LiquidFuelStorage,LiquidFuelStorageComplex,50.755962,-1.292791,2,LFS01
4,TEL08,Telecomms,TelecommunicationsExchange,50.730509,-1.164576,3,NaN


In [12]:
new_dependencies.head(10)

,from_asset,to_asset,connection,dependency_score
0,PGS09,LFS02,PGS09-->LFS02,3
1,PGS09,LFS01,PGS09-->LFS01,3
2,PGS09,LFS03,PGS09-->LFS03,3
3,PGS09,TEL08,PGS09-->TEL08,3


In [ ]:
def add_new_assets_to_graph(G, assets_df):
    """
    Adds new assets from a DataFrame to an existing graph.

    Each row in the DataFrame is treated as a new asset. The 'Asset_ID'
    is used as the node ID, and all other columns in the row are added
    as attributes to that node.

    Args:
        G (nx.DiGraph): The existing NetworkX directed graph to add assets to.
        assets_df (pd.DataFrame): A DataFrame containing the new assets.
                                  Must include an 'Asset_ID' column.

    Returns:
        nx.DiGraph: The graph with the new assets added as nodes.
    """
    if 'Asset_ID' not in assets_df.columns:
        raise ValueError("assets_df must contain an 'Asset_ID' column.")

    for index, row in assets_df.iterrows():
        # Use the 'Asset_ID' as the node identifier
        node_id = row['Asset_ID']

        # Convert the entire row to a dictionary to use as attributes
        attributes = row.to_dict()

        # Add the node to the graph with all its attributes
        if not G.has_node(node_id):
            G.add_node(node_id, **attributes)
        else:
            # If the node already exists, update its attributes
            nx.set_node_attributes(G, {node_id: attributes})

    #print(f"Added {len(G.number_of_nodes)} new nodes to the graph.")
    return G

In [25]:
G =  add_new_assets_to_graph(G, new_assets)

In [26]:
G.number_of_nodes()

133

In [10]:
G.nodes['PGS09']

{'Asset_ID': 'PGS09',
 'Asset_Type': 'PowerGen/Substation',
 'Asset_Name': 'LowVoltageElectricitySubstationComplex',
 'lat': 50.703494,
 'long': -1.2920646,
 'criticality_score': 3,
 'backup_asset_id': nan}

In [27]:
G = add_new_dependency_to_graph(G, new_dependencies)

In [28]:
G.number_of_edges()

51